In [1]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password="swapnil@123",
    host="127.0.0.1",
    port=5432,
    database="bluestock_mf"
)

engine = create_engine(url)

In [2]:
connection = engine.connect()

print("Connected Successfully!")

connection.close()

Connected Successfully!


In [3]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

In [4]:
fund_master = pd.read_sql("SELECT * FROM fund_master", engine)

scheme_performance = pd.read_sql(
    "SELECT * FROM scheme_performance", engine
)

investor_transactions = pd.read_sql(
    "SELECT * FROM investor_transactions", engine
)

portfolio_holdings = pd.read_sql(
    "SELECT * FROM portfolio_holdings", engine
)

monthly_sip = pd.read_sql(
    "SELECT * FROM monthly_sip", engine
)

industry_folio = pd.read_sql(
    "SELECT * FROM industry_folio", engine
)

In [5]:
fund_master['launch_date'] = pd.to_datetime(fund_master['launch_date'])

In [6]:
fund_master['fund_age_years'] = (
    pd.Timestamp.today() - fund_master['launch_date']
).dt.days / 365

In [7]:
fund_master[['scheme_name','fund_age_years']].head()

,scheme_name,fund_age_years
0,SBI Bluechip Fund - Regular Plan - Growth,20.369863
1,SBI Bluechip Fund - Direct Plan - Growth,13.484932
2,SBI Small Cap Fund - Regular Plan - Growth,16.800000
3,SBI Small Cap Fund - Direct Plan - Growth,13.484932
4,SBI Magnum Gilt Fund - Regular Plan - Growth,25.498630


In [8]:
scheme_performance['excess_return_3yr'] = (
    scheme_performance['return_3yr_pct']
    - scheme_performance['benchmark_3yr_pct']
)

In [9]:
scheme_performance[
['return_3yr_pct',
'benchmark_3yr_pct',
'excess_return_3yr']
].head()

,return_3yr_pct,benchmark_3yr_pct,excess_return_3yr
0,12.36,11.49,0.87
1,11.30,9.52,1.78
2,23.39,22.16,1.23
3,23.14,22.01,1.13
4,6.07,4.47,1.60


In [10]:
scheme_performance['risk_adjusted_return'] = (
    scheme_performance['return_5yr_pct']
    / scheme_performance['std_dev_ann_pct']
)

In [11]:
scheme_performance[
['return_5yr_pct',
'std_dev_ann_pct',
'risk_adjusted_return']
].head()

,return_5yr_pct,std_dev_ann_pct,risk_adjusted_return
0,14.45,14.0,1.032143
1,14.23,14.0,1.016429
2,20.67,25.0,0.826800
3,21.82,25.0,0.872800
4,5.43,4.0,1.357500


In [12]:
investor_transactions['transaction_date'] = pd.to_datetime(
    investor_transactions['transaction_date']
)

In [14]:
investor_transactions['transaction_year'] = (
    investor_transactions['transaction_date'].dt.year
)

In [15]:
investor_transactions['transaction_month'] = (
    investor_transactions['transaction_date'].dt.month
)

In [16]:
investor_transactions['income_segment'] = np.where(
    investor_transactions['annual_income_lakh']>=20,
    'High Income',
    'Regular Income'
)

In [17]:
investor_transactions[
['annual_income_lakh','income_segment']
].head()

,annual_income_lakh,income_segment
0,77.1,High Income
1,7.1,Regular Income
2,47.2,High Income
3,54.4,High Income
4,14.5,Regular Income


In [18]:
portfolio_holdings['holding_value_pct'] = (
    portfolio_holdings['market_value_cr']
    /
    portfolio_holdings.groupby('amfi_code')
    ['market_value_cr']
    .transform('sum')
)*100

In [19]:
monthly_sip['mom_growth_pct'] = (
    monthly_sip['sip_inflow_crore']
    .pct_change()*100
)

In [20]:
industry_folio['equity_share_pct'] = (
    industry_folio['equity_folios_crore']
    /
    industry_folio['total_folios_crore']
)*100

In [21]:
fund_master.to_sql(
    "fund_master_features",
    engine,
    if_exists="replace",
    index=False
)

scheme_performance.to_sql(
    "scheme_performance_features",
    engine,
    if_exists="replace",
    index=False
)

investor_transactions.to_sql(
    "investor_transactions_features",
    engine,
    if_exists="replace",
    index=False
)

portfolio_holdings.to_sql(
    "portfolio_holdings_features",
    engine,
    if_exists="replace",
    index=False
)

monthly_sip.to_sql(
    "monthly_sip_features",
    engine,
    if_exists="replace",
    index=False
)

industry_folio.to_sql(
    "industry_folio_features",
    engine,
    if_exists="replace",
    index=False
)

21